## **選擇條件** (80 min)
### **讓程式具有選擇和判斷的能力**
###   **1. 比較運算子**

| 意義 | 等於 | 不等於| 大於 | 大於等於 | 小於 | 小於等於 |
|---|---|---|---|---|---|---|
|運算符號| == | != | > | >= | < | <= |

<br>

#### **注意事項**
*   運算子需要相連，不能空格分開
```python
3 ! = 4 (X)
3 != 4 (O)
```
*   == 與 = 是不同的運算
  * == 是比較運算子，會判斷左右兩側的資料是否相
  * = 是賦值計算，會將右側的資料代入左側的資料


###  **2. 邏輯運算子**

| 意義 | 且 | 或 | 非 |
|---|---|---|---|
|運算式| a = x and y | a = x or y | a = not x |



---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("APCS_4-1")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / APCS_4-1.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

<br>

### **APCS 4-1 布林值**
#### **題目說明：**
#### 給定 a, b, c, d, e 五個變數和相關運算程式碼
* 請輸出 a, b, c 的資料型態
* 請計算 d, e 的運算結果

``` python
#start

x = True
y = False

a = x and y

b = 11
c = 5.5

d = b < c
e = b >= c*2
```

#### **解題想法**
* 使用 type() 回傳的輸出 <class ‘資料型態’>

In [ ]:
#start-APCS_4-1


In [ ]:
make_submit_button("APCS_4-1")


<br>

### **APCS 4-2 成績及格**

#### **條件判斷式寫法**
``` python
#基本型
if a > b:
  # 執行的程式
#其他程式
```
```python
#基本型2
if a > b:
  # 執行的程式
else:
  # 執行的程式
#其他程式
```
```python
#基本型3
if a > b:
  # 執行的程式1
elif a == b:
  # 執行的程式2
else:
  # 執行的程式3
#其他程式
```

#### **題目說明：**
#### 成績系統除了分數以外還需要記錄是否通過該科目，老師可以依照考試難易度設定及格門檻，如果大於或等於及格門檻則是通過，反之則是未通過

#### **測資說明：**
*   第一個數字為及格門檻
*   之後有多個數字為分數
*   輸出則為小寫的 pass/fail

#### **此題目的學習要點：**
*   條件式語法與格式
*   連續 IPO

In [ ]:
#start-APCS_4-2


In [ ]:
make_submit_button("APCS_4-2")


<br>

### **APCS 4-3 奇偶數 (自我挑戰)**
#### **題目說明：**
#### 數字系統中，整數的部分有分成奇數和偶數兩種特別的群體，請設計一個程式，輸入指定的正整數後，判斷這個數字是奇數還是偶數。
#### **輸入說明:**
*   0 以外的正整數

#### **輸出說明:**
*   若為奇數輸出 odd，若為偶數輸出 even





#### **解題想法**

*   下面為需要撰寫的程式流程項目註解:

    ```python
    #start
    #輸入數字

    #判斷是奇數還是偶數
    
    #輸出 odd, even
    ```
*   需要使用條件式
*   需要使用連續 IPO
*   對題目沒頭緒的時候想想解題心法

In [ ]:
#start-APCS_4-3


In [ ]:
make_submit_button("APCS_4-3")


<br>

### **APCS 4-4 三角形**
#### **題目說明：**
#### 要能組成一個三角形，三個邊長一定要滿足任兩邊總和大於第三邊這個條件，現在有三根不同長度的木頭 (a, b, c)，請寫一個程式判斷是否能組合成三角形

#### **輸入說明：**
*   輸入有若干列
*   每一列有三個數字，分別代表 a, b, c

#### **輸出說明：**
*   若這三個數字能形成三角形
*   輸出 Yes, 否則輸出 No


#### **解題想法**
```python
#輸入
一共有三個數字, 而且在同一行

#計算
任兩邊相加 > 第三邊

#輸出
Yes, No
```
#### 此題需要注意的重點
*   每一行有三個數字，使用 split()
*   任兩邊相加 a+b, b+c, c+a
*   輸出結果
*   條件式的變形「複合條件式」

In [ ]:
#start-APCS_4-4


In [ ]:
make_submit_button("APCS_4-4")


<br>

### **APCS 4-5 分數轉換 (自我挑戰)**
#### **題目說明：**
#### 學校為了避免大家對於成績分分計較，因此決定將期末成績從滿分 100 分轉換為 A, B, C 三種代號表示
*   若分數 80 分以上，則為 A
*   若分數介於 60 分至 79 分，則為 B
*   未滿 60 分是不及格，則為 C





#### **解題想法：**
*   使用條件式判斷輸入分數的數值
*   條件判斷的結果有三個 (A, B, C)，因此需要用到條件式基本型3
*   範例輸入為連續輸入的資料，記得要轉換為連續 IPO 的程式架構

In [ ]:
#start-APCS_4-5


In [ ]:
make_submit_button("APCS_4-5")


<br>

### **APCS 4-6 判斷閏年 (自我挑戰)**
#### **題目說明：**
#### 閏年是比普通年份多出一段時間的年份，在各種曆法都有出現，為的是彌補人類規定的紀年和地球公轉產生的差異
#### 閏年的規則如下：
*   西元年份除以 4 不可整除，則為平年
*   西元年份除以 4 可整除，而且除以 100 不可整除，則為閏年
*   西元年份除以 100 可整除，且除以 400 不可整除，則為平年
*   西元年份除以 400 可整除，則為閏年

#### **輸入說明：**
*   每一列有一個數字代表西元年分

#### **輸出說明：**
*   是閏年輸出 Yes
*   是平年輸出 No



#### **解題想法：**
*   透過題目的閏年條件，整理出程式判斷的規則
*   閏年輸入的範圍為 1600 ~ 3000 年
*   如何判斷是否整除，使用 % 的運算



In [ ]:
#start-APCS_4-6


In [ ]:
make_submit_button("APCS_4-6")


<br>

## 重點複習

*   程式中有流程控制的功能，而選擇條件則是其中一種用法
*   選擇判斷主要依靠「比較運算子」和「邏輯運算子」兩種
*   type() 函式可以知道數值或是變數的資料型態
*   條件式有 3 種基本型的變化，關鍵字為 if, elif, else
*   複合條件式可以混合比較運算和邏輯運算一起使用